# Assignment 02 - Named Entity Recognition (NER) from a News Article

**Author:** Vishal Sigdel

**Goal:** find every named entity in an English news article, label it with its
entity type, export the pairs to a CSV file, and summarise which types appear.

**Pipeline:** read text -> normalise characters -> run the spaCy pipeline -> extract entities -> export -> summarise

| File | Role |
| --- | --- |
| `news.txt` | source article (plain text) |
| `VishalSigdel_NER_01.ipynb` | this notebook |
| `VishalSigdel_NER_01.csv` | output: `Entity`, `Entity_Type` |

## 1. Imports and the language model

`uv run python -m spacy download en_core_web_sm` to install the model

`en_core_web_sm` is the small English pipeline. Its `ner` component is a transition-based
model that labels spans of tokens, not single tokens - which is why `New York Times` comes
back as one entity rather than three.

In [2]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")

print("Pipeline components:", nlp.pipe_names)

Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## 2. Load the article

In [3]:
ARTICLE_PATH = "news.txt"

with open(ARTICLE_PATH, "r", encoding="utf-8") as file:
    news = file.read()

print(f"Characters read: {len(news)}\n")
print(news[:200])

Characters read: 7817

Broad Peak avalanche wipes out a generation of Nepali climbing greats

The deaths of Nirmal Purja and five other mountain guides on Pakistan’s Broad Peak have dealt a severe blow to Nepal’s high-altit


## 3. Normalise the text

News sites use "smart" typography. Curly apostrophes make the tokenizer split
possessives oddly, so `Pakistan's` can lose its entity boundary. Accents are folded too,
keeping entity strings in the CSV plain ASCII and comparable.

Unlike Assignment 01, the mapping is applied *before* the pipeline runs - spaCy's tokenizer
and the NER model both read the same normalised string, so character offsets stay aligned.

In [4]:
import unicodedata

# Curly punctuation -> ASCII equivalent
PUNCTUATION_MAP = {
    "\u2019": "'",   # right single quote
    "\u2018": "'",   # left single quote
    "\u201c": '"',   # left double quote
    "\u201d": '"',   # right double quote
    "\u2013": "-",   # en dash
    "\u2014": "-",   # em dash
    "\u00a0": " ",   # non-breaking space
}


def remove_accents(text):
    """Return `text` with accents stripped (café -> cafe)."""
    decomposed = unicodedata.normalize("NFKD", text)
    return "".join(c for c in decomposed if not unicodedata.combining(c))


def normalise(text):
    """Fold accents and replace curly punctuation with ASCII."""
    text = remove_accents(text)
    for original, replacement in PUNCTUATION_MAP.items():
        text = text.replace(original, replacement)
    return text


news = normalise(news)
print(news[:200])

Broad Peak avalanche wipes out a generation of Nepali climbing greats

The deaths of Nirmal Purja and five other mountain guides on Pakistan's Broad Peak have dealt a severe blow to Nepal's high-altit


## 4. Run the spaCy pipeline

Calling `nlp(text)` runs every component in `nlp.pipe_names` and returns a `Doc`.
The recognised entities land in `doc.ents` as `Span` objects - each one knows its text,
its label, and where it sits in the original string.

In [5]:
doc = nlp(news)

print("Tokens:", len(doc))
print("Entities found:", len(doc.ents))
print()

for entity in doc.ents[:15]:
    print(f"{entity.text:<30} {entity.label_:<10} chars {entity.start_char}-{entity.end_char}")

Tokens: 1451
Entities found: 200

Broad Peak                     PERSON     chars 0-10
Nepali                         GPE        chars 47-53
Nirmal Purja                   ORG        chars 85-97
five                           CARDINAL   chars 102-106
Pakistan                       GPE        chars 132-140
Broad Peak                     PERSON     chars 143-153
Nepal                          PERSON     chars 182-187
decades                        DATE       chars 232-239
Pakistan                       GPE        chars 319-327
8,051-metre                    QUANTITY   chars 330-341
Broad Peak                     PERSON     chars 342-352
Thursday                       DATE       chars 356-364
morning                        TIME       chars 365-372
six                            CARDINAL   chars 397-400
Nepali                         NORP       chars 401-407


## 5. Extract entities and their types

`entity.label_` is the human-readable type; `spacy.explain` turns it into a definition.
The article is about a mountaineering accident, so the types below dominate:

| Type | Meaning |
| --- | --- |
| `PERSON` | people, including fictional |
| `ORG` | companies, agencies, institutions |
| `GPE` | countries, cities, states |
| `LOC` | non-GPE locations - mountains, ranges, bodies of water |
| `DATE` | absolute or relative dates and periods |
| `CARDINAL` | numerals not covered by another type |
| `NORP` | nationalities, religious or political groups |

Entity text is passed through `strip()` because a span can include trailing whitespace
when the model ends an entity at a line break.

In [6]:
extracted_entities = [
    [entity.text.strip(), entity.label_]
    for entity in doc.ents
    if entity.text.strip()
]

print("Entities extracted:", len(extracted_entities))
print(extracted_entities[:20])

Entities extracted: 200
[['Broad Peak', 'PERSON'], ['Nepali', 'GPE'], ['Nirmal Purja', 'ORG'], ['five', 'CARDINAL'], ['Pakistan', 'GPE'], ['Broad Peak', 'PERSON'], ['Nepal', 'PERSON'], ['decades', 'DATE'], ['Pakistan', 'GPE'], ['8,051-metre', 'QUANTITY'], ['Broad Peak', 'PERSON'], ['Thursday', 'DATE'], ['morning', 'TIME'], ['six', 'CARDINAL'], ['Nepali', 'NORP'], ['one', 'CARDINAL'], ['decades', 'DATE'], ['Nirmal Purja', 'PERSON'], ['Nimsdai', 'ORG'], ['five', 'CARDINAL']]


## 6. Build the DataFrame and export

Columns are named `Entity` and `Entity_Type` as the assignment requires, and
`index=False` keeps pandas' row numbers out of the CSV.

In [7]:
df = pd.DataFrame(extracted_entities, columns=["Entity", "Entity_Type"])

df.head(10)

,Entity,Entity_Type
0,Broad Peak,PERSON
1,Nepali,GPE
2,Nirmal Purja,ORG
3,five,CARDINAL
4,Pakistan,GPE
5,Broad Peak,PERSON
6,Nepal,PERSON
7,decades,DATE
8,Pakistan,GPE
9,"8,051-metre",QUANTITY


In [8]:
OUTPUT_PATH = "VishalSigdel_NER_01.csv"

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")

Saved 200 rows to VishalSigdel_NER_01.csv


## 7. Summary of entity types

Two views of the same result. The first counts every *occurrence*, so a name mentioned
eight times contributes eight rows - that is what the CSV holds. The second counts
*distinct* strings per type, which answers "how many different people are named?".

In [9]:
print("Total entity occurrences:", len(df))
print("Distinct entity strings:  ", df["Entity"].nunique())
print("Entity types present:     ", df["Entity_Type"].nunique())
print()

summary = pd.DataFrame({
    "Occurrences": df["Entity_Type"].value_counts(),
    "Distinct": df.groupby("Entity_Type")["Entity"].nunique(),
})
summary["Meaning"] = [spacy.explain(t) for t in summary.index]

summary

Total entity occurrences: 200
Distinct entity strings:   117
Entity types present:      12



,Occurrences,Distinct,Meaning
Entity_Type,,,
CARDINAL,23,13,Numerals that do not fall under another type
DATE,35,31,Absolute or relative dates or periods
GPE,33,16,"Countries, cities, states"
LAW,2,2,Named documents made into laws.
LOC,10,1,"Non-GPE locations, mountain ranges, bodies of ..."
NORP,10,6,Nationalities or religious or political groups
ORDINAL,8,5,"""first"", ""second"", etc."
ORG,20,15,"Companies, agencies, institutions, etc."
PERSON,49,28,"People, including fictional"


### Most frequent entities

Which specific names carry the story.

In [10]:
top_entities = (
    df.groupby(["Entity", "Entity_Type"])
    .size()
    .sort_values(ascending=False)
    .head(15)
    .rename("Mentions")
)

top_entities.reset_index()

,Entity,Entity_Type,Mentions
0,Everest,LOC,10
1,Broad Peak,PERSON,5
2,Nepal,PERSON,5
3,Kanchenjunga,GPE,4
4,Manaslu,GPE,4
5,Yukta,GPE,4
6,Seven,CARDINAL,4
7,Nimsdai,ORG,4
8,Pakistan,GPE,4
9,Nawang Thendu Sherpa,PERSON,4


### Distinct entities grouped by type

In [11]:
for entity_type in summary.index:
    names = sorted(df.loc[df["Entity_Type"] == entity_type, "Entity"].unique())
    print(f"{entity_type} ({len(names)}): {', '.join(names)}")
    print()

CARDINAL (13): 10, 12, 14, K2, Seven, Six, dozens, five, four, one, only 29, six, three

DATE (31): 1980, 1996, 2002, 2011, 2015, 2018, 2021, 2022, 29-year-old, Earlier this year, Friday, January 16, 2021, July 3, Less than a month, May, May 20 this year, Only weeks, Three years ago, Thursday, a single season, decades, earlier this year, nearly eight years, the age of 16, the age of 22, the first winter, the following year, the years, this year, three weeks earlier, years

GPE (16): Annapurna, Beding, Dolakha, Gorkha, Kanchenjunga, Laprak, Makalu, Makalu Rural, Makalu Rural Municipality, Manaslu, Nepal, Nepali, Nima, Okhaldhunga, Pakistan, Yukta

LAW (2): Everest 10 times, Everest 15

LOC (1): Everest

NORP (6): Chinese, Himalayan, Manaslu, Nepali, Omani, Russian

ORDINAL (5): first, second, seventh, tenth, third

ORG (15): Chyakse Boda, Dawa Chirring Sherpa, Dharche Rural, Imagine Nepal, Kilu, Laprak, Lobuche, Nimsdai, Nirmal Purja, Ongdi, Ongdi Tshering Sherpa, Pur Bahadur Gurung, Ru

## 8. Bonus - visualise the entities

`displacy` renders the article with each entity boxed and labelled in place, which makes
mislabelled spans easy to spot by eye. Only the first few sentences are rendered - the
whole article would be a wall of colour.

In [12]:
from spacy import displacy

# First 5 sentences, re-parsed so displacy gets a Doc with its own offsets
excerpt = " ".join(sentence.text for sentence in list(doc.sents)[:5])

displacy.render(nlp(excerpt), style="ent", jupyter=True)